<a href="https://colab.research.google.com/github/JMTolbert2005/Bone-Fracture-data-detection/blob/main/TeamSEVEN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
print("Hello MLSN Team 7!")

Hello MLSN Team 7!


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install ultralytics

In [ ]:
# Load and import all necessary

from google.colab import drive # Google Drive to directly full data from drive/colab
import matplotlib.pyplot as plt # Useful Library for creating graphs
from ultralytics import YOLO
import pandas as pd #Necessary for all tibble & datasets
import numpy as np # Useful Library for creating arrays
import zipfile as zp # Useful library to unzip files
import os # Interactive way to work with the underbelly of a project, any directory work is done through here
from collections import defaultdict
import shutil  # More Interactive ways to work with file directories
import cv2

from IPython.display import Image, display # Allows for image display
from google.colab.patches import cv2_imshow


# 1. Mount the Shared Drive; # This asks for permission to see the Team folder and WILL make a pop up, please click yes! :)
drive.mount('/content/drive')

# 2. Define where the ZIP is and where you want it to go: Please Update this match the exact

# Where the zip is goes here <- Important
zip_path_on_drive = '/content/drive/Shareddrives/MLSN 07/TEAM-07/Data/archive(1).zip'

#Where you want the zip to go (on local colab) goes here
colab_path = '/content/ExampleHere2'

# 3. Effeciency if-else to be able to
if not os.path.exists(colab_path):
    print("Copying zip from Drive to Colab...")
    # We copy it first because unzipping directly from Drive is slow
    shutil.copy(zip_path_on_drive, '/content/temp.zip')
    print("Zip Copying Done!")

    #Begin the actual unzip
    print("Unzipping images...")
    with zp.ZipFile('/content/temp.zip', 'r') as zp:
        zp.extractall(colab_path)
    print("Unzip Done!")

    print(f"Success! Images are ready at: {colab_path}")

    # Remove former temporary zip
    os.remove('/content/temp.zip')
else:
    print("Images are already unzipped and ready!")

Mounted at /content/drive
Copying zip from Drive to Colab...
Zip Copying Done!
Unzipping images...
Unzip Done!
Success! Images are ready at: /content/ExampleHere2


In [ ]:
from PIL import Image

#E1: Did not Initialize or synxtax error
#E2 Needed to tab
#E3: Logic issue, with the tabs off and newpath not defined the function could remove images that were not added properly

def ensure_jpg_and_labels(images_dir, labels_dir):
  images = os.listdir(images_dir)
  labels = os.listdir(labels_dir)

  removed = 0
  converted = 0



  image_basenames=set() #2
  label_basenames=set(l.replace('.txt','')for l in labels if l.endswith('.txt')) #2

  print(f"   [In {images_dir.split('/')[-2]}] Found {len(images)} images and {len(labels)} labels.") #Added to make sure code works


  for image in images: #2
    img_path=os.path.join(images_dir,image) #2
    name, ext=os.path.splitext(image) #img isn't defined you put it as [Iamge] #2
    ext=ext.lower() #2

    if ext not in ['.jpg','.jpeg','.png']: #2'
      continue #2'

    if ext != '.jpg': #2'
      new_path=os.path.join(images_dir,name+'.jpg') #2'
      Image.open(img_path).convert('RGB').save(new_path) #2' & #3, OS.Rename might crash so repalced with open and conert shorthand
      converted += 1
      image_basenames.add(name) #2'



    else:
      new_path=img_path
      image_basenames.add(name)

    if name not in label_basenames: #2'
      os.remove(new_path) #2'
      removed += 1 #1, #2''

  for lbl in labels: #2
    name=lbl.replace('.txt','') #2
    if name not in image_basenames: #2
      os.remove(os.path.join(labels_dir,lbl)) #2
      removed += 1 #1 & #2'

  print(f'Converted {converted} images, removed {removed} images')

In [ ]:
base="/content/ExampleHere2/BoneFractureYolo8"
for split in ["train","valid","test"]:
  ensure_jpg_and_labels(f"{base}/{split}/images",f"{base}/{split}/labels")

   [In train] Found 3631 images and 3631 labels.
Converted 0 images, removed 0 images
   [In valid] Found 348 images and 348 labels.
Converted 0 images, removed 0 images
   [In test] Found 169 images and 169 labels.
Converted 0 images, removed 0 images


In [ ]:
# Interactive verison that is also a function demo

def interactive_file_count():

  filepath = input("Please enter your filepath:")

  print(len(os.listdir(filepath)))

interactive_file_count()

Please enter your filepath:8


FileNotFoundError: [Errno 2] No such file or directory: '8'

In [ ]:
def displayimagebynum():

  num = int(input("Please input a number"))
  filepath = input("Please enter your filepath:")
  display(Image(filename =f'{filepath}/{os.listdir(filepath)[num]}'))

displayimagebynum()



In [ ]:
def plot_im_box(image, labels, class_names=None, color=(0,255,0), thickness=2):

    # If labels is a file path, load it
    if isinstance(labels, str):
        if not os.path.exists(labels):
            raise FileNotFoundError(labels)
        with open(labels, "r") as f:
            labels = f.readlines()

    h, w = image.shape[:2]

    for line in labels:
        parts = line.strip().split()
        if len(parts) < 7:
            continue

        class_id = int(parts[0])
        coords = list(map(float, parts[1:]))

        # Convert yolo polygon coords
        xs = np.array(coords[0::2]) * w
        ys = np.array(coords[1::2]) * h

        x1, y1 = int(xs.min()), int(ys.min())
        x2, y2 = int(xs.max()), int(ys.max())

        # Draw rectangle
        cv2.rectangle(image, (x1,y1), (x2,y2), color, thickness)

        # Label
        label = class_names[class_id]


        cv2.putText(
            image, label,
            (x1, max(y1 - 6, 12)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6, color, 2
        )

    return image


img_path = input("Enter File Path:")
image = cv2.imread(img_path)

label_path = img_path.replace("/images/", "/labels/").replace(".jpg", ".txt")

with open(label_path) as f:
    labels = f.readlines()

class_names = ['elbow positive', 'fingers positive', 'forearm fracture', 'humerus fracture', 'humerus', 'shoulder fracture', 'wrist positive'] # Need to eradicate humerous fracture to "null" or "background"

out = plot_im_box(image, labels, class_names)
cv2_imshow(out)
cv2.waitKey(0)
cv2.destroyAllWindows()

In [ ]:

!cat /content/ExampleHere2/BoneFractureYolo8/train/labels/image1_1138_png.rf.6b2a58c4f3569c908d988b1379cc0541.txt

In [ ]:


# Check contents of the 'train' directory
train_path = '/content/ExampleHere2/BoneFractureYolo8/train'
print(f"\nContents of {train_path}:")
if os.path.exists(train_path):
    for item in os.listdir(train_path):
        print(os.path.join(train_path, item))
else:
    print(f"Directory {train_path} does not exist.")

# Check contents of the 'test' directory
test_path = '/content/ExampleHere2/BoneFractureYolo8/test'
print(f"\nContents of {test_path}:")
if os.path.exists(test_path):
    for item in os.listdir(test_path):
        print(os.path.join(test_path, item))
else:
    print(f"Directory {test_path} does not exist.")

In [ ]:
from collections import defaultdict


def check_class_balance(labels_folder):
    class_counts = defaultdict(int)
    MODIFIED = 0 # Swap to modified instead of DL


    for filename in os.listdir(labels_folder):  # Nornal Filepath
        filepath = os.path.join(labels_folder, filename)
        with open(filepath, "r") as f:
           lines = f.readlines()

        newlines = []
        for line in lines:
            if line.strip(): # Strip the line completely
              class_id = line.split()[0] #Look for each single line
              if class_id in ['0', '1', '2','4', '5', '6']:
                newlines.append(line)

        if len(newlines) < len(lines): # Write everything back as normal
            with open(filepath, "w") as f:
                f.writelines(newlines)
                MODIFIED += 1

    print(f"Modified {MODIFIED} Files")


    for file in os.listdir(labels_folder):
        if file.endswith(".txt"):
            file_path = os.path.join(labels_folder, file)

            with open(file_path, "r") as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) == 0:
                        continue

                    class_id = int(parts[0])
                    class_counts[class_id] += 1

    return class_counts


labels_folder = input("Enter label Path:")

class_counts = check_class_balance(labels_folder)

print("Class Counts:", dict(class_counts))

classes = sorted(class_counts.keys())
counts = [class_counts[c] for c in classes]#

plt.figure()
plt.bar(classes, counts)
plt.xlabel("Class ID")
plt.ylabel("Number of Labels")
plt.title("Bone Class Distribution")
plt.show()

In [ ]:
!cat /content/ExampleHere2/BoneFractureYolo8/test/labels/image1_1588_png.rf.aba29f401dee4db13fc5434f3c0186b4.txt

In [ ]:
import os
from collections import defaultdict

# List contents of the base directory where data was unzipped
base_unzip_path = '/content/ExampleHere2'
print(f"Contents of {base_unzip_path}:")
if os.path.exists(base_unzip_path):
    for item in os.listdir(base_unzip_path):
        print(os.path.join(base_unzip_path, item))
else:
    print(f"Directory {base_unzip_path} does not exist.")

# Also check the expected sub-path for 'BoneFractureYolo8'
expected_data_path = '/content/ExampleHere2/BoneFractureYolo8'
print(f"\nContents of {expected_data_path}:")
if os.path.exists(expected_data_path):
    for item in os.listdir(expected_data_path):
        print(os.path.join(expected_data_path, item))
else:
    print(f"Directory {expected_data_path} does not exist.")

def preprocess_images(filepath): #This function seems unneeded
  IMGPROTO = cv2.imread(filepath)

  processed_img = np.zeros_like(IMGPROTO)
  processed_img[:, :, 2] = IMGPROTO[:, :, 2]

  print((IMGPROTO.shape, processed_img.shape))

  return processed_img

filepath = preprocess_images('/content/ExampleHere2/BoneFractureYolo8/train/images/distal-humeral-fractures-2-_JPEG.rf.39ba3f53047ecb4064613850a1c76fc1.jpg')

plt.imshow(cv2.cvtColor(filepath, cv2.COLOR_BGR2RGB))
plt.axis('off')
plt.show()

In [ ]:
img_exampke = cv2.imread("/content/ExampleHere2/BoneFractureYolo8/test/images/distal-humerus-fracture-1_jpg.rf.831cb137cfcbde1079f86abd5f5f2867.jpg")


fig, ax = plt.subplots(figsize=(10, 10))
ax.imshow(img_exampke)
ax.axis('off')
plt.show()

In [ ]:
import os
import hashlib

BASE_PATH = '/content/ExampleHere2/BoneFractureYolo8'
Train_PATH = os.path.join(BASE_PATH, 'train', 'images') #All 3 f string were raplced with os.path.join to properly call; will explain during meeting
Valid_PATH = os.path.join(BASE_PATH, 'valid', 'images')
Test_PATH = os.path.join(BASE_PATH, 'test', 'images')

def image_hash(filepath):
  with open (filepath, 'rb') as f:
    return hashlib.md5(f.read()).hexdigest()

def collect_hashes(images_dir):
  hashes={}
  for images_name in os.listdir(images_dir):
    img_path = os.path.join(images_dir, images_name)
    if os.path.isfile(img_path):
      hashes[images_name] = image_hash(img_path) #SInce Hashes was already set a dictionary simply got rid of the recalling here
  return hashes #Major Indentation error, before this is effectively only running once as return is force inside the loop


train_hashes = collect_hashes(Train_PATH)
valid_hashes = collect_hashes(Valid_PATH)
test_hashes = collect_hashes(Test_PATH)

train_val_dupes = set(train_hashes.values()) & set(valid_hashes.values()) #Values to See x3 VVV
train_test_dupes = set(train_hashes.values()) & set(test_hashes.values())
valid_test_dupes = set(valid_hashes.values()) & set(test_hashes.values())

print ("Duplicate check results")
print (f"Train and Validation duplicates: {len(train_val_dupes)}")
print (f"Train and Test duplicates: {len(train_test_dupes)}")
print (f"Validation and Test duplicates: {len(valid_test_dupes)}")

In [ ]:
import os

def get_file_class(labels_folder, target_class=6):
    matching_files = []

    images_folder = labels_folder.replace("/labels", "/images")

    for file in os.listdir(labels_folder):
        if file.endswith(".txt"):
            file_path = os.path.join(labels_folder, file)

            with open(file_path, "r") as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) == 0:
                        continue

                    if int(parts[0]) == target_class:
                        image_file = file.replace(".txt", ".jpg")
                        image_path = os.path.join(images_folder, image_file)
                        matching_files.append(image_path)
                        break

    return matching_files


class_file = get_file_class(
    "/content/ExampleHere2/BoneFractureYolo8/train/labels"
)
print(class_file)


In [ ]:
# Model Transform
vals = ['0', '1', '-1']

def ezimgflip(path):
    imgtru = cv2.imread(path)
    base_dir = os.path.dirname(path)
    base_name = os.path.splitext(os.path.basename(path))[0]

    output_dir = os.path.join(base_dir, "flipped")
    os.makedirs(output_dir, exist_ok=True)

    for i in vals:
        imgflip = cv2.flip(imgtru, int(i))
        out_path = os.path.join(output_dir, f"{base_name}_flip_{i}.jpg")
        cv2.imwrite(out_path, imgflip)
        fig, ax = plt.subplots(figsize=(10, 10))
        ax.imshow(imgflip)
        ax.axis('off')
        print(f"Saved flipped image on {i}: {out_path}")
        plt.show()




In [ ]:
for i in range(len(class_file)):
  if i <= 10:
    ezimgflip(class_file[i])


In [ ]:
# Example Class Remove


for folder in folders:
    for filename in os.listdir(folder):
        filepath = os.path.join(folder, filename)

        # Read 1st chararacter  If '3', delete and index
        DELFILE = 0
        if open(filepath).read(1) == '3':
            os.remove(filepath) and DELFILE += 1
            print(f"Deleted {filepath}")

        print(f"The Amount of deleted files is {DELFILE}")


In [ ]:
def plot_im_box(image, labels, class_names=None, color=(0,255,0), thickness=2):

    # If labels is a file path, load it
    if isinstance(labels, str):
        if not os.path.exists(labels):
            raise FileNotFoundError(labels)
        with open(labels, "r") as f:
            labels = f.readlines()

    h, w = image.shape[:2]

    for line in labels:
        parts = line.strip().split()
        if len(parts) < 7:
            continue

        class_id = int(parts[0])
        coords = list(map(float, parts[1:]))

        # Convert yolo polygon coords
        xs = np.array(coords[0::2]) * w
        ys = np.array(coords[1::2]) * h

        x1, y1 = int(xs.min()), int(ys.min())
        x2, y2 = int(xs.max()), int(ys.max())

        # Draw rectangle
        cv2.rectangle(image, (x1,y1), (x2,y2), color, thickness)

        # Label
        label = class_names[class_id]


        cv2.putText(
            image, label,
            (x1, max(y1 - 6, 12)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6, color, 2
        )

    return image


img_path = input("Enter File Path:")
image = cv2.imread(img_path, 1)

label_path = img_path.replace("/images/", "/labels/").replace(".jpg", ".txt")

with open(label_path) as f:
    labels = f.readlines()

class_names = ['elbow positive', 'fingers positive', 'forearm fracture', 'humerus fracture', 'humerus', 'shoulder fracture', 'wrist positive']

out = plot_im_box(image, labels, class_names)
cv2_imshow(out)
cv2.waitKey(0)
cv2.destroyAllWindows()

In [ ]:
#Begin Model Testing   #Step 3

# Get the actual model
model = YOLO('yolov8n.pt') # Using Nano Yolo; perhaps Medium would fit best?

results = model.train(data='/content/ExampleHere2/BoneFractureYolo8/data.yaml',      # your data.yaml path
                      epochs=150,  # Number of full training cycles
                      imgsz=640,  # Image size 640 x 640
                      batch=16, #Images to process at once
                      name = 'quick_test') #optimizer='AdamW'

# Validate on validation set
# Wil return Mean average precision, precision, and recall
metrics = model.val()

model.predict(
    source='/content/ExampleHere2/BoneFractureYolo8/test/images',
    save=True,
)


print(f"Results saved to: runs/detect/quick_test, predictions in runs/detect/predict")

resultsreal = pd.read_csv("/content/runs/detect/quick_test/results.csv")

resultsreal.head()

In [ ]:
!nvidia-smi #step 2

In [ ]:
# PRE-CHECK
import os
from pathlib import Path

print("=== PRE-FLIGHT CHECK ===\n")

# 1. Check data.yaml exists
data_yaml = '/content/ExampleHere2/BoneFractureYolo8/data.yaml'
if os.path.exists(data_yaml):
    print(f"✓ data.yaml found")
    # Show what's in it
    with open(data_yaml) as f:
        print(f.read())
else:
    print(f"✗ ERROR: data.yaml NOT FOUND at {data_yaml}")

# 2. Check train/val/test folders exist and have images
for split in ['train', 'valid', 'test']:
    img_folder = f'/content/ExampleHere2/BoneFractureYolo8/{split}/images'
    lbl_folder = f'/content/ExampleHere2/BoneFractureYolo8/{split}/labels'

    if os.path.exists(img_folder):
        img_count = len([f for f in os.listdir(img_folder) if f.endswith(('.jpg', '.png'))])
        lbl_count = len([f for f in os.listdir(lbl_folder) if f.endswith('.txt')])
        print(f"✓ {split}: {img_count} images, {lbl_count} labels")
    else:
        print(f"✗ ERROR: {split} folder missing")

# 3. Test YOLO import
try:
    from ultralytics import YOLO
    print("\n✓ YOLOv8 library installed")
except:
    print("\n✗ ERROR: Install with: pip install ultralytics")

# 4. Test model download
try:
    model = YOLO('yolov8n.pt')
    print("✓ Model loaded successfully")
except Exception as e:
    print(f"✗ Model load error: {e}")

print("\n=== ALL CHECKS PASSED - READY FOR DEMO ===")

In [ ]:
!pip uninstall -y albumentations

In [ ]:
#Fortuh Iteration - CHosen Final Model
# Get the actual model


model = YOLO('yolov8m.pt') # Using Nano Yolo; perhaps Medium would fit best?

results = model.train(data='/content/ExampleHere2/BoneFractureYolo8/data.yaml',
                      epochs=200,  # Number of full training cycles
                      imgsz=640,  # Image size 640 x 640
                      batch=16, #Images to process at once
                      project ='/content/runs/detect',
                      patience = 15,
                      name = 'FinalModel') #optimizer='AdamW'

# Validate on validation set
# Wil return Mean average precision, precision, and recall
metrics = model.val()

model.predict(
    source='/content/ExampleHere2/BoneFractureYolo8/test/images',
    save=True,
    project = '/content/runs/detect',
    name = 'FinalModel_predict'
)


print(f"Results saved to google drive")
shutil.copytree('/content/runs/detect/FinalModel', '/content/drive/Shareddrives/MLSN 07/TEAM-07/finmod2', dirs_exist_ok=True)
shutil.copytree('/content/runs/detect/FinalModel_predict', '/content/drive/Shareddrives/MLSN 07/TEAM-07/finmod2', dirs_exist_ok=True)
print("Results saved to google drive")

realres = pd.read_csv("/content/runs/detect/FinalModel")

realres.head()

#Results here:

In [ ]:
# Iteration 3

!pip uninstall -y albumentations
# Get the actual model
model = YOLO('yolov8m.pt') # Using Nano Yolo; perhaps Medium would fit best?

results = model.train(data='/content/ExampleHere2/BoneFractureYolo8/data.yaml',      # your data.yaml path
                      epochs=100,  # Number of full training cycles
                      imgsz=640,  # Image size 640 x 640
                      batch=16, #Images to process at once
                      name = 'iteration3',
                      augment=False,
                      mosaic = 0.0,
                      lr0 = 0.001,
                      lrf = 0.01,
                      momentum = 0.9,
                      weight_decay = 0.005,
                      patience = 20,
                      optimizer='AdamW',) #optimizer='AdamW'

# Validate on validation set
# Wil return Mean average precision, precision, and recall
metrics = model.val()

model.predict(
    source='/content/ExampleHere2/BoneFractureYolo8/test/images',
    save=True,
)


print(f"Results saved to: runs/detect/iteration3, predictions in runs/detect/predict")

resultit3 = pd.read_csv("/content/runs/detect/iteration3/results2.csv")

resultit3.head()


In [ ]:
# Iteration 3

!pip uninstall -y albumentations
# Get the actual model
model = YOLO('yolov8m.pt') # Using Nano Yolo; perhaps Medium would fit best?

results = model.train(data='/content/ExampleHere2/BoneFractureYolo8/data.yaml',      # your data.yaml path
                      epochs=100,  # Number of full training cycles
                      imgsz=1040,  # Image size 640 x 640
                      batch=8, #Images to process at once
                      name = 'iteration3',
                      ##
                      box = 20.0,
                      cls = 4.0,
                      ##
                      augment=False,
                      blur=0
                      mosaic = 0.0,
                      lr0 = 0.001,
                      lrf = 0.01,
                      momentum = 0.9,
                      weight_decay = 0.005,
                      patience = 20,
                      optimizer='AdamW',) #optimizer='AdamW'

# Validate on validation set
# Wil return Mean average precision, precision, and recall
metrics = model.val()

model.predict(
    source='/content/ExampleHere2/BoneFractureYolo8/test/images',
    save=True,
)


print(f"Results saved to: runs/detect/iteration3, predictions in runs/detect/predict")

resultit3 = pd.read_csv("/content/runs/detect/iteration3/results2.csv")

resultit3.head()

In [ ]:
# Iteration 4 - Now using new roboflow set with background images capped at 25%


# Get the actual model
model = YOLO('yolov8m.pt')

results = model.train(data='/content/ExampleHere3/data.yaml',
                      epochs=60,  # Number of full training cycles
                      batch=8, #Images to process at once
                      name = 'iteration4',
                      augment=False,
                      auto_augment = False,
                      mosaic = 0.0,
                      lr0 = 0.001,
                      lrf = 0.01,
                      momentum = 0.9,
                      weight_decay = 0.005,
                      patience = 20,
                      optimizer='AdamW',) #optimizer='AdamW'

# Validate on validation set
# Wil return Mean average precision, precision, and recall
metrics = model.val()

model.predict(
    source='/content/ExampleHere3/test',
    save=True,
)


print(f"Results saved to: runs/detect/iteration3, predictions in runs/detect/predict")

resultit3 = pd.read_csv("/content/runs/detect/iteration3/results2.csv")

resultit3.head()

In [ ]:
# Iteration 5 - Now using new roboflow set with background images capped at 3%


# Get the actual model
model = YOLO('yolov8m.pt')

results = model.train(data='/content/ExampleHere5/data.yaml',
                      epochs=60,  # Number of full training cycles
                      batch=16, #Images to process at once
                      name = 'iteration4',
                      augment=False,
                      auto_augment = False,
                      mosaic = 0,
                      patience = 10
                    ) #optimizer='AdamW'

# Validate on validation set
# Wil return Mean average precision, precision, and recall
metrics = model.val()

model.predict(
    source='/content/ExampleHere5/test/images',
    save=True,
)


print(f"Results saved to: runs/detect/iteration3, predictions in runs/detect/predict")

resultit3 = pd.read_csv("/content/runs/detect/iteration3/results2.csv")

resultit3.head()

In [ ]:
# Iteraiton 6

# Get the actual model
model = YOLO('yolov8m.pt')

results = model.train(task = 'detect',
mode = 'train',
                      data='/content/ExampleHere6/data.yaml',
                      epochs=60,  # Number of full training cycles
                      batch=16, #Images to process at once
                      name = 'iteration6',
                      augment=False,
                      auto_augment = False,
                      mosaic = 0,
                      patience = 10
                    ) #optimizer='AdamW'

# Validate on validation set
# Wil return Mean average precision, precision, and recall
metrics = model.val()

model.predict(
    source='/content/ExampleHere6/test/images',
    save=True,
)


print(f"Results saved to: runs/detect/iteration3, predictions in runs/detect/predict")

resultit3 = pd.read_csv("/content/runs/detect/iteration3/results2.csv")

resultit3.head()

In [ ]:
# All fracture test
# Iteraiton 7

# Get the actual model
model = YOLO('yolov8m.pt')

results = model.train(task = 'detect',
mode = 'train',
                      data='/content/ExampleHere7/data.yaml',
                      epochs=100,  # Number of full training cycles
                      batch=16, #Images to process at once
                      name = 'iteration6',
                      augment=False,
                      auto_augment = False,
                      mosaic = 0,
                      patience = 20
                    ) #optimizer='AdamW'

# Validate on validation set
# Wil return Mean average precision, precision, and recall
metrics = model.val()

model.predict(
    source='/content/ExampleHere7/test/images',
    save=True,
)


print(f"Results saved to: runs/detect/iteration3, predictions in runs/detect/predict")

resultit3 = pd.read_csv("/content/runs/detect/iteration3/results2.csv")

resultit3.head()

In [ ]:
# Large Yolo
# Iteraiton 7

# Get the actual model
model = YOLO('yolov81.pt')

results = model.train(task = 'detect',
mode = 'train',
                      data='/content/ExampleHere6/data.yaml',
                      epochs=60,  # Number of full training cycles
                      batch=16, #Images to process at once
                      name = 'iteration6',
                      augment=False,
                      auto_augment = False,
                      mosaic = 0,
                      patience = 10
                    ) #optimizer='AdamW'

# Validate on validation set
# Wil return Mean average precision, precision, and recall
metrics = model.val()

model.predict(
    source='/content/ExampleHere6/test/images',
    save=True,
)


print(f"Results saved to: runs/detect/iteration3, predictions in runs/detect/predict")

resultit3 = pd.read_csv("/content/runs/detect/iteration3/results2.csv")

resultit3.head()

In [ ]:
# Extra Large Yolo
# Iteraiton 8 EXTRA LARGE

# Get the actual model
model = YOLO('yolov8x.pt')

results = model.train(task = 'detect',
                      mode = 'train',
                      data='/content/ExampleHere6/data.yaml',
                      imgsz = 1056,
                      epochs=100,  # Number of full training cycles
                      batch=8, #Images to process at once
                      name = 'iteration6',
                      augment=False,
                      auto_augment = False,
                      mosaic = 0,
                      patience = 20
                    )

# Validate on validation set
# Wil return Mean average precision, precision, and recall
metrics = model.val()

model.predict(
    source='/content/ExampleHere6/test/images',
    save=True,
)


print(f"Results saved to: runs/detect/iteration3, predictions in runs/detect/predict")

resultit3 = pd.read_csv("/content/runs/detect/iteration3/results2.csv")

resultit3.head()

Ultralytics 8.4.19 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=False, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/ExampleHere6/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1056, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8x.pt, momentum=0.937, mosaic=0, multi_scale=0.0, name=iteration6, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=20, perspective=

RuntimeError: Dataset '/content/ExampleHere6/data.yaml' error ❌ '/content/ExampleHere6/data.yaml' does not exist

In [ ]:
# 1. Install the UI library and YOLO
!pip install gradio ultralytics
from ultralytics import YOLO
import gradio as gr


modelrun = YOLO('/content/drive/Shareddrives/MLSN 07/TEAM-07/finmod2/weights/best.pt')

def fracturepredict(image):
    # Run YOLO prediction on the uploaded image
    results = modelrun(image)

    # Plot the bounding boxes onto the image
    annoimg = results[0].plot()

    # Gradio requires RGB format, OpenCV uses BGR under the hood
    return cv2.cvtColor(annoimg, cv2.COLOR_BGR2RGB)

# 3. Create the instant frontend template
Live = gr.Interface(
    fn=fracturepredict,

    inputs=gr.Image(type="numpy", label="Upload Image here (X-ray)"),

    outputs=gr.Image(type="numpy", label="Model Prediction for fracture location"),

    title="MLSN Bone Fracture Detector",

    description="Drag and drop an X-Ray image with a fracture to see where the fracture is!"
)

# 4. Launch the web UI directly inside Colab
Live.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://24609f958e11cfccf1.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)



0: 608x640 1 forearm fracture, 1 humerus, 17.3ms
Speed: 20.9ms preprocess, 17.3ms inference, 26.6ms postprocess per image at shape (1, 3, 608, 640)
